In [1]:
using DifferentialEquations # for the actual time evolution
using OrdinaryDiffEq # for ODEs
using Plots # for plotting
using Base.Threads # for parallelization
using StaticArrays # somehow needed to use multiple variables in DifferentialEquations.jl

using Plots, LaTeXStrings, Colors
using Plots.PlotMeasures
using LinearAlgebra

using Random, Distributions

using FFTW # discrete Fourier transform

using JLD2 # for file saving

In [2]:
level = "../../../../"

include(joinpath(level, "src/4th-order-FD-stencils.jl"));
include(joinpath(level, "src/evolution_Liouville_larger_cutoff.jl"));
include(joinpath(level, "src/hamiltonian_Liouville.jl"));
include(joinpath(level, "src/initial_data_moving_gaussian.jl"));
include(joinpath(level, "src/visualisation.jl"));

### evolution

In [3]:
function artisan_evolution_at_resolution(Nx, stableRandomSeed, pModel, pInit, target_time)
    # unpack model parameters
    (mphi2, mchi2, c4, c, epsDiss) = pModel;
    
    # set the spatial discretization
    NboundaryPadding = 2;#Int(div(Nx,2));  # Number of boundary padding points
    dx = 1/(Nx);  # Grid spacing
        
    pGrid = (dx, Nx, NboundaryPadding);
    # reset a combined set of parameters (residual from old structure ... could be modified)
    # TODO: modify to pGrid, pModel, pInit
    p = (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding);

    # set the time span
    tspan = (0, target_time);

    # set the evolution method
    time_integration_method = RK4();
    
    # generate initial conditions
    u0 = initial_data(
        range(0, step=dx, length=(Nx + 2 * NboundaryPadding)),
        p, 
        pInit
    );
        
    # set the problem
    prob = ODEProblem(finite_differenced_pde_with_bc!, u0, tspan, p);

    sol = solve(
        prob, time_integration_method, 
        saveat = tspan[end]/10^3, #exp.(range(log(tspan[1]), log(tspan[end]), length=10^4))
        dt=dx/10, 
        adaptive = false, 
        dense=false, 
        maxiters=typemax(Int),
        callback=field_size_callback
    );
        
    # obtain the hamiltonian
    hamiltonian = zeros(length(sol.u))
    hamphi = zeros(length(sol.u))
    hamchi = zeros(length(sol.u))
    for i = 1:length(sol.u)
        hamiltonian[i] = nintegrate_simps(hamiltonian_density(sol.u[i], p), dx)
        hamphi[i] = nintegrate_simps(hamiltonian_phi(sol.u[i], p), dx)
        hamchi[i] = nintegrate_simps(hamiltonian_chi(sol.u[i], p), dx)
    end
    
    return (p, sol, hamiltonian, hamphi, hamchi)
end

artisan_evolution_at_resolution (generic function with 1 method)

In [4]:
function evolution_at_amplitude(amp, current_target_time, current_res_log2, stableRandomSeed)
    
    #stableRandomSeed = 42
    print("persistent random seed: ", stableRandomSeed, "\n")
    
    print("current characteristic amplitude: ", amp, "\n")
    
    # set monitoring flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # parameters of the model
    mphi2 = 1.;
    mchi2 = 15 * mphi2;
    c4 = 1.0;
    c = -1.;
    epsDiss = 0;

    # parameters of the initial data
    a0phi = + amp; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    a0chi = a0phi; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    k0phi = 2^2;
    k0chi = k0phi;
    x0phi = 0.3;
    x0chi = 0.7;

    offsetphi = sqrt(((mphi2 - mchi2)^2 - 8 * mphi2 * c4) / (32 * c4^2))
    offsetchi = - sqrt(((mphi2 - mchi2)^2 - 8 * mchi2 * c4) / (32 * c4^2))

    aStochastic = 0;
    mink = 1;
    maxk = 4;
    
    desiredTkinPhi = NaN;
    desiredTkinChi = NaN;
    
    # set the combined set of parameters 
    pModel = (mphi2, mchi2, c4, c, epsDiss);
    pInit = (
        a0phi, a0chi, k0phi, k0chi, x0phi, x0chi, 
        offsetphi, offsetchi, 
        aStochastic, mink, maxk, stableRandomSeed,
        desiredTkinPhi, desiredTkinChi
    );
     
    # set some tables to store intermediate output
    resTab = [2^i for i in current_res_log2-2:current_res_log2]
    pTab = []
    solTab = []
    hamiltonianTab = []
    hamPhiTab = []
    hamChiTab = []
    
    #############################
    # evolution
    #############################
    
    # run evolution
    for res in resTab
        print("current resolution: ", res, "\n")
        # run the evolution
        @time (p, sol, hamiltonian, hamPhi, hamChi) = artisan_evolution_at_resolution(
            res, stableRandomSeed, pModel, pInit, current_target_time
        )
        print("... terminated", "\n")
        # unpack parameters
        (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding) = p
        # append the results
        push!(pTab, p)
        push!(solTab, sol)
        push!(hamiltonianTab, hamiltonian)
        push!(hamPhiTab, hamPhi)
        push!(hamChiTab, hamChi)
    end
    
    #############################
    # CONVERGENCE
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",amp)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        print("Output plot directory does not exist. Creating it ...\n")
        mkpath(dir_path)
    else
        print("Output plot directory already exists.\n")
    end
    
    # plot and determine convergence 
    loss_of_convergence_time = save_convergence_plots(
        resTab, pTab, solTab, 
        hamiltonianTab, 
        dir_path
    )
    if loss_of_convergence_time >= solTab[end].t[end]
        print("Convergence kept at all times.\n")
    else
        print("Convergence lost at time t=",loss_of_convergence_time,"\n")
    end
    
    # determine the index of convergence loss
    loss_of_convergence_index = findfirst(t -> t > loss_of_convergence_time, solTab[end].t)
    if loss_of_convergence_index === nothing
        loss_of_convergence_index = length(solTab[end].t)
    end
    
    #print(10 * hamPhiTab[end][1],"\n")
    #print(hamPhiTab[end][2:10],"\n")
    
    # determine the onset time of the runaway (e-fold increase in either kinetic energy)
    runaway_index_phi = findfirst(
        energy -> abs(energy) > exp(1) * maximum(abs.(hamPhiTab[end][1:Int(div(length(hamPhiTab[end]),4))+1])), 
        hamPhiTab[end]
    )
    if runaway_index_phi === nothing
        runaway_index_phi = length(hamPhiTab[end])
    end
    runaway_index_chi = findfirst(
        energy -> abs(energy) > exp(1) * maximum(abs.(hamChiTab[end][1:Int(div(length(hamChiTab[end]),4))+1])), 
        hamChiTab[end]
    )
    if runaway_index_chi === nothing
        runaway_index_chi = length(hamChiTab[end])
    end
    runaway_index = min(runaway_index_phi, runaway_index_chi)
    runaway_time = solTab[end].t[runaway_index]
    if runaway_time >= solTab[end].t[end]
        print("No runaway detected.\n")
    else
        print("Runaway detected at time t=",runaway_time,"\n")
    end
    
    #############################
    # GENERATE REMAINING PLOTS IF DESIRED
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",amp)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        #print("Directory does not exist. Creating it...")
        mkpath(dir_path)
    end
        
    # plot energy components
    save_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_normalised_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_difference_in_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    
    # plot field heatmaps
    save_density_plots(
        solTab[end], pTab[end], pInit,
        dir_path,
        loss_of_convergence_time=loss_of_convergence_time
    )
    
    # save snapshots
    save_snaps(
        solTab[end], pTab[end];
        snap_intervals=Int(round(length(solTab[end])/1)), 
        yrangeVal=1.2,
        dir_path = dir_path
    );
    
    # animate the fields
    save_animation(
        solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
        pTab[end],
        join([dir_path, "/animation_Nx=", resTab[end], ".gif"])
    );    
#     # animate frequencies
#     save_animation_momentum_space(
#         solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
#         pTab[end],
#         join([dir_path, "/animation_momentum_space_Nx=", resTab[end], ".gif"])
#     );
    
    print("Finished plotting.", "\n")
    
    #############################
    # SAVE DATA
    #############################
    
    if runaway_time > loss_of_convergence_time
        print("WARNING: Convergence not maintained until onset of runaway. Resolution insufficient.", "\n")
    else
        convergence_maintained = true
        if runaway_time >= solTab[end].t[end]
            print("WARNING: Lower bound only because target time insufficient.", "\n")
        else
            lower_bound_only = false
        end
    end
    
    dir_path = string("dat/",stableRandomSeed)
    if !isdir(dir_path)
        mkpath(dir_path)
    end

    timesteps = solTab[end].t
    stable_until = min(runaway_time, loss_of_convergence_time)

    @save joinpath(pwd(), dir_path, string(amp,".jld2")) amp stable_until lower_bound_only timesteps hamiltonianTab hamPhiTab hamChiTab

    print("Saved data.", "\n")

    
    return (runaway_time, convergence_maintained, lower_bound_only)
end

evolution_at_amplitude (generic function with 1 method)

### main()

In [5]:
function main()
    
    # some random seed (can be modified at will)
    stableRandomSeed = 0#rand(1:10^7)
    
    # initialise flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # set abort criteria ...
    highest_res_log2 = 13;
    max_target_time = 2 * 10^4;
    # ... and their initial values
    current_res_log2 = 11;
    current_target_time = 10;
    
    # initialise the runaway time for handover to next amplitude
    runaway_time = Inf;
    
    # set table of desired amplitudes (NOTE: links to scaling assumption below)
    amp_base = 1
    amplitudes = [invAmp for invAmp in 0.5:0.5:16]
    amplitudes = amplitudes.^(-1)
    
    # loop over all amplitudes
    for amp in amplitudes
        
        # re-attempt while flags not positive or until abort criteria met
        while (!convergence_maintained||lower_bound_only) && (current_res_log2 <= highest_res_log2) && (current_target_time <= max_target_time)
            # attempt run and obtain flags
            (runaway_time, convergence_maintained, lower_bound_only) = evolution_at_amplitude(
                amp, 
                current_target_time, 
                current_res_log2,
                stableRandomSeed
            )
            # update according to obtained flags
            if convergence_maintained                
                if lower_bound_only
                    current_target_time = current_target_time * 4
                    print("Increasing target time to T = ", current_target_time, "\n")
                else
                    current_target_time = min(runaway_time, current_target_time);
                    print("Target time reset to confidently detected runaway time T = ", current_target_time, "\n")
                end
            else
                current_res_log2 = current_res_log2 + 1;
                print("Increasing resolution from N = ", current_res_log2 - 1, " to ", current_res_log2, "\n")
            end
        end
        
        print("AMPLITUDE A = ", amp, " DONE!\n")
        
        # update target time based on the presumed scaling assumption and adapt the target time accordingly
        current_target_time = runaway_time
        print("Updating target time for next amplitude from T = ", current_target_time, " ... ")
        current_target_time = current_target_time * exp(amp_base)     
        print("to T = ", current_target_time, "\n")
        
        # decrease resolution if convergence was maintained in previous step
#         if convergence_maintained
#             current_res_log2 = current_res_log2 - 1;
#             print("Decreasing resolution from N = ", current_res_log2 + 1, " to ", current_res_log2, "\n")
#         end
        
        # check whether it makes sense to go on; otherwise abort 
        if convergence_maintained && lower_bound_only && current_target_time >= max_target_time
            print("ABORT: maximum target time approached in converged simulation; no use to proceed")
            return
        end
        
        # ensure that current params don't exceed the abort criteria for the next step
        current_res_log2 = min(current_res_log2, highest_res_log2)
        current_target_time = min(current_target_time, max_target_time)
        
        # reset the flags
        convergence_maintained = false;
        lower_bound_only = true;
    end

end

main (generic function with 1 method)

In [6]:
main()

persistent random seed: 0
current characteristic amplitude: 2.0
current resolution: 256
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.9996145057343142
		max |amplitude| chi before rescaling: 1.9996145057343142
  6.192698 seconds (5.00 M allocations: 2.853 GiB, 4.95% gc time, 73.32% compilation time)
... terminated
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.9996145057343142
		max |amplitude| chi before rescaling: 1.9996145057343142
  5.065318 seconds (3.73 M allocations: 9.955 GiB, 7.51% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.9999759044312804
		max |amplitude| chi before rescaling: 1.9999759044312804
 18.173979 seconds (7.42 M allocations: 38.654 GiB, 10.96% gc time)
... terminated
Output plot directory already exists.
Convergence lost at time t=9.36
Runaway detected at time t=6.19
Finished plotting.


[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/01_Gauss/plots/0/2.0/animation_Nx=1024.gif


Saved data.
Target time reset to confidently detected runaway time T = 6.19
AMPLITUDE A = 2.0 DONE!
Updating target time for next amplitude from T = 6.19 ... to T = 16.82616451816149
persistent random seed: 0
current characteristic amplitude: 1.0
current resolution: 256
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.9998072528671571
		max |amplitude| chi before rescaling: 0.9998072528671571
  3.708624 seconds (3.05 M allocations: 4.420 GiB, 20.70% gc time, 2.40% compilation time: 13% of which was recompilation)
... terminated
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.9998072528671571
		max |amplitude| chi before rescaling: 0.9998072528671571
  9.924937 seconds (6.25 M allocations: 16.688 GiB, 7.83% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.9999879522156402
		max |amplitude| chi before rescaling: 0.9999879522156402
 35.936220 seconds 

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/01_Gauss/plots/0/1.0/animation_Nx=1024.gif


  9.320031 seconds (6.25 M allocations: 16.688 GiB, 7.45% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.9999879522156402
		max |amplitude| chi before rescaling: 0.9999879522156402
 32.042804 seconds (12.45 M allocations: 64.919 GiB, 6.25% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.9999879522156402
		max |amplitude| chi before rescaling: 0.9999879522156402
122.876130 seconds (35.89 M allocations: 255.207 GiB, 5.52% gc time)
... terminated
Output plot directory already exists.
Convergence kept at all times.
No runaway detected.
Finished plotting.


...

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/01_Gauss/plots/0/0.09523809523809523/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 125.07031249872783.
 84.870175 seconds (46.13 M allocations: 123.409 GiB, 9.75% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.09090799565596729
		max |amplitude| chi before rescaling: 0.09090799565596729
Terminating because one of the fields grew too large at time t = 125.47343750292082.
297.853370 seconds (92.54 M allocations: 482.874 GiB, 9.42% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.09090799565596729
		max |amplitude| chi before rescaling: 0.09090799565596729
Terminating because one of the fields grew too large at time t = 124.70122070819221.
1095.238428 seconds (265.65 M allocations: 1.845 TiB, 9.20% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence lost at time t=121.8086351074292
Runaway detected at time t=91.94435584363863
F

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/01_Gauss/plots/0/0.09090909090909091/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 131.17773437387015.
 70.180241 seconds (48.38 M allocations: 129.433 GiB, 10.21% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.08695547410570784
		max |amplitude| chi before rescaling: 0.08695547410570784


LoadError: InterruptException:

### export .jl for production run

In [2]:
using NBInclude
nbexport("main.jl", "main.ipynb")